# 05. EDA: 폭염일 vs 평소

**목표**: 본격 모델링 전에 "폭염일에 매출이 달라지는가, 강남과 춘천이 다르게 반응하는가"를 빠르게 확인

**읽는 것**: `analysis_visitor.csv`, `analysis_time.csv` (04 실행 후)

**기간**: 7-9월만 사용 (10-12월은 폭염이 없음)

## 1. 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/bigcontest-2026")
OUT = BASE / "data/processed"
FIG = BASE / "outputs/figures"
FIG.mkdir(parents=True, exist_ok=True)

In [ ]:
# 그래프 한글 폰트 (나눔고딕 설치 후 등록)
!apt-get -qq install -y fonts-nanum > /dev/null
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
av = pd.read_csv(OUT / "analysis_visitor.csv", parse_dates=["date"])
at = pd.read_csv(OUT / "analysis_time.csv", parse_dates=["date"])

av = av[av["summer"] == 1]
at = at[at["summer"] == 1]
print(av.shape, at.shape)
print(av.groupby("region")["heat"].agg(["sum", "count"]))

## 2. 기준 매출과 매출 비율

요일 효과를 빼야 폭염 효과가 보입니다. 같은 지역·업종·**요일**의 **비폭염·비공휴일** 날 중앙값을 "평소 매출"로 두고,

`ratio = 그날 매출 ÷ 평소 매출`

ratio가 0.9면 평소보다 10% 적게 팔린 날입니다. (EDA용 간이 기준이고, 정식 기대 매출 모델은 06에서 만듭니다.)

In [ ]:
def add_ratio(df, keys):
    base = (df[(df["heat"] == 0) & (df["holiday"] == 0)]
            .groupby(keys + ["dow"])["sales"].median().rename("base"))
    df = df.merge(base, on=keys + ["dow"], how="left")
    df["ratio"] = df["sales"] / df["base"]
    return df

# 업종 단위 (외지인 + 지역민 합산)
grp = (av.groupby(["date", "region", "ind_grp", "dow", "heat", "holiday", "tmax", "heat_streak"])
         ["sales"].sum().reset_index())
grp = add_ratio(grp, ["region", "ind_grp"])
grp = grp[grp["holiday"] == 0]       # 광복절 제외
print(grp["ratio"].describe())

## 3. 일별 매출과 최고기온 흐름

전체 매출과 기온이 같이 움직이는지 눈으로 먼저 확인

In [ ]:
daily_tot = grp.groupby(["region", "date"]).agg(sales=("sales", "sum"), tmax=("tmax", "first")).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, r in zip(axes, ["강남", "춘천"]):
    d = daily_tot[daily_tot["region"] == r]
    ax.plot(d["date"], d["sales"] / 1e8, color="tab:blue")
    ax.set_ylabel("매출 (억 원)")
    ax2 = ax.twinx()
    ax2.plot(d["date"], d["tmax"], color="tab:red", alpha=0.6)
    ax2.axhline(33, color="tab:red", ls="--", lw=0.8)
    ax2.set_ylabel("최고기온 (℃)")
    ax.set_title(f"{r}: 일 매출(파랑)과 최고기온(빨강)")
plt.tight_layout()
plt.savefig(FIG / "05_daily_sales_tmax.png", dpi=150)
plt.show()

## 4. 핵심: 폭염일 매출 비율 (업종 × 지역)

폭염일 평균 ratio와 평소 날 평균 ratio를 비교하고, Welch t-검정으로 차이가 우연인지 확인합니다.

**effect**가 -10%면 폭염일 매출이 평소보다 약 10% 낮다는 뜻.

In [ ]:
from scipy import stats

rows = []
for (r, g), d in grp.groupby(["region", "ind_grp"]):
    h, n = np.log(d.loc[d["heat"] == 1, "ratio"]), np.log(d.loc[d["heat"] == 0, "ratio"])
    t, p = stats.ttest_ind(h, n, equal_var=False)
    rows.append({"region": r, "ind_grp": g,
                 "effect": np.exp(h.mean() - n.mean()) - 1, "p": p})
eff = pd.DataFrame(rows)

table = eff.pivot(index="ind_grp", columns="region", values="effect")
pvals = eff.pivot(index="ind_grp", columns="region", values="p")
table = table.sort_values("춘천")
show = table.map("{:+.1%}".format) + pvals.map(lambda p: " *" if p < 0.05 else "")
print("폭염일 매출 변화 (* = p<0.05)")
print(show.to_string())

In [ ]:
ax = table.plot.barh(figsize=(8, 7), color={"강남": "tab:blue", "춘천": "tab:orange"})
ax.axvline(0, color="black", lw=0.8)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:+.0%}"))
ax.set_title("폭염일(33℃ 이상) 매출 변화: 업종 × 지역")
ax.set_xlabel("평소 대비 매출 변화")
plt.tight_layout()
plt.savefig(FIG / "05_heat_effect_by_industry.png", dpi=150)
plt.show()

In [ ]:
import statsmodels.formula.api as smf

# 폭염일이 어느 달에 몰려 있는지 (휴가철 쏠림 확인)
day = grp.drop_duplicates(["date", "region"])
print(pd.crosstab([day["region"], day["date"].dt.month], day["heat"]))

# 주차 + 요일을 통제한 폭염 효과
grp["lsales"] = np.log(grp["sales"])
grp["week"] = grp["date"].dt.isocalendar().week.astype(int)

rows = []
for (r, g), d in grp.groupby(["region", "ind_grp"]):
    m = smf.ols("lsales ~ heat + C(week) + C(dow)", data=d).fit(cov_type="HC1")
    rows.append({"region": r, "ind_grp": g,
                 "effect": np.exp(m.params["heat"]) - 1, "p": m.pvalues["heat"]})
fe = pd.DataFrame(rows)

t = fe.pivot(index="ind_grp", columns="region", values="effect")
p = fe.pivot(index="ind_grp", columns="region", values="p")
print("\n폭염일 매출 변화 - 주차·요일 통제 (* = p<0.05)")
print((t.map("{:+.1%}".format) + p.map(lambda x: " *" if x < 0.05 else "")).to_string())

In [ ]:
import statsmodels.formula.api as smf
RAW_CARD = BASE / "data/raw/card"

check = ["백화점", "쇼핑몰", "면세점", "기타유통", "전용매장", "체인점",   # 대형유통
         "종합레저타운/놀이동산", "실내/실외골프장"]                        # 야외레저

raw = pd.read_csv(RAW_CARD / "신한카드_빅콘테스트2026_데이터2.txt", sep="\t",
                  encoding="cp949", dtype=str,
                  usecols=["TA_YMD", "MCT_SGG_CD", "MCT_RY_CD", "TS_AT"])
raw = raw[raw["MCT_RY_CD"].isin(check)]
raw["date"] = pd.to_datetime(raw["TA_YMD"], format="%Y%m%d")
raw["region"] = raw["MCT_SGG_CD"].map({"서울 강남구": "강남", "강원 춘천시": "춘천"})
raw["sales"] = pd.to_numeric(raw["TS_AT"])

ind = raw.groupby(["date", "region", "MCT_RY_CD"])["sales"].sum().reset_index()
days = grp[["date", "region", "heat", "dow", "week", "holiday"]].drop_duplicates()
ind = ind.merge(days, on=["date", "region"])          # 여름·비공휴일만 남음
ind["lsales"] = np.log(ind["sales"])

rows = []
for (r, i), d in ind.groupby(["region", "MCT_RY_CD"]):
    if len(d) < 60:                                    # 데이터 부족한 업종 제외
        continue
    m = smf.ols("lsales ~ heat + C(week) + C(dow)", data=d).fit(cov_type="HC1")
    rows.append({"region": r, "industry": i, "days": len(d),
                 "share_of_sales": d["sales"].sum(),
                 "effect": np.exp(m.params["heat"]) - 1, "p": m.pvalues["heat"]})
res = pd.DataFrame(rows)
res["share_of_sales"] = res.groupby("region")["share_of_sales"].transform(lambda s: s / s.sum())
print(res.sort_values(["region", "share_of_sales"], ascending=[True, False])
         .assign(effect=lambda x: x["effect"].map("{:+.1%}".format),
                 share_of_sales=lambda x: x["share_of_sales"].map("{:.0%}".format),
                 p=lambda x: x["p"].round(3))
         .to_string(index=False))

## 5. 기온 구간별 매출 비율

33℃ 하나로 자르지 않고 기온을 구간으로 나눠, 몇 도부터 꺾이는지 감을 잡습니다. (임계온도 분석의 사전 확인)

In [ ]:
bins = [0, 27, 29, 31, 33, 35, 40]
labels = ["~27", "27~29", "29~31", "31~33", "33~35", "35~"]
grp["tbin"] = pd.cut(grp["tmax"], bins, labels=labels)

focus = ["음식점", "카페·제과", "식료품소매", "야외레저", "숙박", "의료"]
curve = (grp[grp["ind_grp"].isin(focus)]
         .groupby(["ind_grp", "region", "tbin"], observed=True)["ratio"].mean().reset_index())

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=True)
for ax, g in zip(axes.flat, focus):
    for r, c in [("강남", "tab:blue"), ("춘천", "tab:orange")]:
        d = curve[(curve["ind_grp"] == g) & (curve["region"] == r)]
        ax.plot(d["tbin"].astype(str), d["ratio"], marker="o", color=c, label=r)
    ax.axhline(1, color="gray", lw=0.8)
    ax.set_title(g)
    ax.tick_params(axis="x", rotation=45)
axes[0, 0].legend()
fig.suptitle("최고기온 구간별 매출 비율 (1 = 평소)")
plt.tight_layout()
plt.savefig(FIG / "05_temp_bins.png", dpi=150)
plt.show()

print(grp.groupby(["region", "tbin"], observed=True)["date"].nunique().unstack(0))  # 구간별 일수

## 6. 외지인 vs 지역민

폭염일에 빠지는 게 누구의 소비인지

In [ ]:
vis = av[av["holiday"] == 0].copy()
vis = add_ratio(vis, ["region", "ind_grp", "is_visitor"])

rows = []
for (r, g, v), d in vis.groupby(["region", "ind_grp", "is_visitor"]):
    h, n = np.log(d.loc[d["heat"] == 1, "ratio"]), np.log(d.loc[d["heat"] == 0, "ratio"])
    rows.append({"region": r, "ind_grp": g, "who": "외지인" if v else "지역민",
                 "effect": np.exp(h.mean() - n.mean()) - 1})
ve = pd.DataFrame(rows).pivot_table(index="ind_grp", columns=["region", "who"], values="effect")
print("폭염일 매출 변화: 외지인 vs 지역민")
print(ve.map("{:+.1%}".format).to_string())

# 지역 전체
tot = (vis.groupby(["date", "region", "is_visitor", "heat"])["sales"].sum().reset_index())
print(tot.groupby(["region", "is_visitor", "heat"])["sales"].mean().unstack().assign(
    change=lambda x: x[1] / x[0] - 1)["change"].map("{:+.1%}".format))

## 7. 시간대: 폭염일에 한낮 매출이 저녁으로 옮겨가는가

하루 매출 중 각 시간대가 차지하는 비중을 폭염일과 평소로 비교

In [ ]:
t = at[at["holiday"] == 0]
share = (t.groupby(["date", "region", "ind_grp", "time_gb", "heat"])["sales"].sum()
          .groupby(level=["date", "region", "ind_grp"]).transform(lambda s: s / s.sum())
          .rename("share").reset_index())
sh = share.groupby(["region", "ind_grp", "time_gb", "heat"])["share"].mean().unstack("heat")
sh["diff_pp"] = (sh[1] - sh[0]) * 100       # 퍼센트포인트

print("폭염일 시간대 비중 변화 (%p, 양수 = 폭염일에 비중 증가)")
print(sh["diff_pp"].unstack(["region", "time_gb"]).round(1).to_string())

## 8. 확인 체크리스트

결과를 보고 아래를 판단합니다.

| 질문 | 어디서 보나 | 성립 조건 |
|---|---|---|
| 폭염일에 매출이 달라지는 업종이 있는가 | 4번 표 | 몇 개 업종에서 * 표시 |
| 강남과 춘천의 반응이 다른가 | 4번 그래프 | 같은 업종의 막대 방향·크기가 다름 |
| 꺾이는 온도가 보이는가 | 5번 그래프 | 특정 구간 이후 선이 꺾임 |
| 외지인과 지역민이 다르게 반응하는가 | 6번 표 | 외지인 쪽 변화가 더 큼 |
| 한낮 매출이 저녁으로 이동하는가 | 7번 표 | 12_17 음수, 18_23 양수 |

주의: 7~9월 안에서도 7월(폭염 많음)과 9월(폭염 없음)의 계절 차이가 섞여 있습니다. 정식 분석(06)에서는 주차 효과를 넣어 통제합니다.

## 시간대 이동도 주차 통제로 확인
섹션 7은 단순 비교라 휴가철 효과가 섞였을 수 있어요. 섹션 4처럼 주차·요일을 통제해서 다시 확인해요. 05 노트북 맨 아래에 추가해 주세요.


In [ ]:
import statsmodels.formula.api as smf

s = share.merge(at[["date", "region", "dow"]].drop_duplicates(), on=["date", "region"])
s["week"] = s["date"].dt.isocalendar().week.astype(int)

rows = []
for (r, g, tg), d in s.groupby(["region", "ind_grp", "time_gb"]):
    m = smf.ols("share ~ heat + C(week) + C(dow)", data=d).fit(cov_type="HC1")
    rows.append({"region": r, "ind_grp": g, "time_gb": tg,
                 "diff_pp": m.params["heat"] * 100, "p": m.pvalues["heat"]})
ts = pd.DataFrame(rows)

out = ts.pivot_table(index="ind_grp", columns=["region", "time_gb"], values="diff_pp").round(1)
sig = ts.pivot_table(index="ind_grp", columns=["region", "time_gb"], values="p")
print("폭염일 시간대 비중 변화 - 주차·요일 통제 (%p, * = p<0.05)")
print((out.astype(str) + sig.map(lambda p: "*" if p < 0.05 else "")).to_string())

# 한낮 감소 + 저녁 증가가 동시에 나타난 업종 수
w = ts.pivot_table(index=["region", "ind_grp"], columns="time_gb", values="diff_pp")
print("\n한낮↓ 저녁↑ 업종 수:")
print(((w["12_17"] < 0) & (w["18_23"] > 0)).groupby("region").sum())

## 비를 통제하고, 비 효과도 같이 보기

In [ ]:
import statsmodels.formula.api as smf

wx = av[["date", "region", "rain"]].drop_duplicates()
g2 = grp.merge(wx, on=["date", "region"])
g2["lsales"] = np.log(g2["sales"])
g2["week"] = g2["date"].dt.isocalendar().week.astype(int)
g2["rainy"] = (g2["rain"] >= 1).astype(int)     # 비 온 날
g2["heavy"] = (g2["rain"] >= 30).astype(int)    # 호우 (rainy에 추가로 더해지는 효과)

day = g2.drop_duplicates(["date", "region"])
print(pd.crosstab([day["region"], day["heat"]], [day["rainy"], day["heavy"]]))

rows = []
for (r, g), d in g2.groupby(["region", "ind_grp"]):
    m = smf.ols("lsales ~ heat + rainy + heavy + C(week) + C(dow)", data=d).fit(cov_type="HC1")
    for v in ["heat", "rainy", "heavy"]:
        rows.append({"region": r, "ind_grp": g, "var": v,
                     "eff": np.exp(m.params[v]) - 1, "p": m.pvalues[v]})
res = pd.DataFrame(rows)

tab = res.pivot_table(index="ind_grp", columns=["region", "var"], values="eff")
sig = res.pivot_table(index="ind_grp", columns=["region", "var"], values="p")
print("\n주차·요일·비 통제 후 효과 (* = p<0.05)")
print((tab.map("{:+.1%}".format) + sig.map(lambda p: "*" if p < 0.05 else "")).to_string())